# Cooperative Q-Learning for Multi-Agent Reinforcement Learning

所有智能体共享一个Q表，基于联合状态和团队奖励进行学习


In [ ]:
import numpy as np
import random
from collections import defaultdict


In [ ]:
class CooperativeQLearning:
    """
    合作Q学习算法
    所有智能体共享一个Q表，使用联合状态和团队奖励
    """
    
    def __init__(self, num_agents, state_space_size, action_space_size,
                 learning_rate=0.1, discount_factor=0.95, epsilon_start=1.0,
                 epsilon_end=0.1, epsilon_decay=0.995):
        self.num_agents = num_agents
        self.state_space_size = state_space_size
        self.action_space_size = action_space_size
        self.learning_rate = learning_rate
        self.discount_factor = discount_factor
        self.epsilon = epsilon_start
        self.epsilon_start = epsilon_start
        self.epsilon_end = epsilon_end
        self.epsilon_decay = epsilon_decay
        
        # 共享的Q表
        self.joint_action_space_size = action_space_size ** num_agents
        self.q_table = defaultdict(lambda: np.zeros(self.joint_action_space_size))
        
        # 统计信息
        self.episode_rewards = []
        self.total_updates = 0
    
    def joint_state_to_key(self, joint_state):
        """将联合状态转换为Q表的键"""
        state_tuples = tuple(tuple(s.astype(int)) for s in joint_state)
        return state_tuples
    
    def joint_action_to_index(self, joint_action):
        """将联合动作转换为索引"""
        index = 0
        for i, action in enumerate(joint_action):
            index += action * (self.action_space_size ** (self.num_agents - 1 - i))
        return index
    
    def index_to_joint_action(self, index):
        """将索引转换为联合动作"""
        joint_action = []
        remaining = index
        for i in range(self.num_agents):
            power = self.action_space_size ** (self.num_agents - 1 - i)
            action = remaining // power
            remaining = remaining % power
            joint_action.append(action)
        return tuple(joint_action)
    
    def select_joint_action(self, joint_state, training=True):
        """选择联合动作（ε-贪婪策略）"""
        joint_state_key = self.joint_state_to_key(joint_state)
        
        if training and random.random() < self.epsilon:
            joint_action = tuple(random.randint(0, self.action_space_size - 1) 
                                for _ in range(self.num_agents))
        else:
            q_values = self.q_table[joint_state_key]
            max_q = np.max(q_values)
            best_indices = np.where(q_values == max_q)[0]
            best_index = int(np.random.choice(best_indices))
            joint_action = self.index_to_joint_action(best_index)
        
        return joint_action
    
    def update(self, joint_state, joint_action, team_reward, next_joint_state, done):
        """更新Q表"""
        joint_state_key = self.joint_state_to_key(joint_state)
        next_joint_state_key = self.joint_state_to_key(next_joint_state)
        joint_action_idx = self.joint_action_to_index(joint_action)
        
        current_q = self.q_table[joint_state_key][joint_action_idx]
        
        if done:
            target_q = team_reward
        else:
            next_max_q = np.max(self.q_table[next_joint_state_key])
            target_q = team_reward + self.discount_factor * next_max_q
        
        new_q = current_q + self.learning_rate * (target_q - current_q)
        self.q_table[joint_state_key][joint_action_idx] = new_q
        
        self.total_updates += 1
    
    def decay_epsilon(self):
        """衰减探索率"""
        if self.epsilon > self.epsilon_end:
            self.epsilon = max(self.epsilon_end, self.epsilon * self.epsilon_decay)
    
    def get_q_value(self, joint_state, joint_action):
        """获取Q值"""
        joint_state_key = self.joint_state_to_key(joint_state)
        joint_action_idx = self.joint_action_to_index(joint_action)
        return self.q_table[joint_state_key][joint_action_idx]
    
    def get_policy(self, joint_state):
        """获取策略（确定性策略：选择Q值最大的联合动作）"""
        joint_state_key = self.joint_state_to_key(joint_state)
        q_values = self.q_table[joint_state_key]
        best_index = int(np.argmax(q_values))
        return self.index_to_joint_action(best_index)
    
    def save_q_table(self, filepath):
        """保存Q表到文件"""
        import pickle
        with open(filepath, 'wb') as f:
            pickle.dump(dict(self.q_table), f)
    
    def load_q_table(self, filepath):
        """从文件加载Q表"""
        import pickle
        try:
            with open(filepath, 'rb') as f:
                self.q_table = defaultdict(lambda: np.zeros(self.joint_action_space_size),
                                           pickle.load(f))
        except FileNotFoundError:
            print(f"Warning: Q-table file {filepath} not found")
